# DPO Pipeline

In [1]:
# Environment Setup
import os
os.environ["CC"] = os.path.expanduser("~/gcc_wrapper")
os.environ.pop("GCC_EXEC_PREFIX", None)
#os.environ["CUDA_VISIBLE_DEVICES"] = "6"
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
#torch.cuda.empty_cache()
#torch.cuda.ipc_collect()

In [2]:
try: import torch
except: raise ImportError('Install torch via `pip install torch`')
from packaging.version import Version as V
v = V(torch.__version__)
cuda = str(torch.version.cuda)
is_ampere = torch.cuda.get_device_capability()[0] >= 8
if cuda != "12.1" and cuda != "11.8" and cuda != "12.4": raise RuntimeError(f"CUDA = {cuda} not supported!")
if   v <= V('2.1.0'): raise RuntimeError(f"Torch = {v} too old!")
elif v <= V('2.1.1'): x = 'cu{}{}-torch211'
elif v <= V('2.1.2'): x = 'cu{}{}-torch212'
elif v  < V('2.3.0'): x = 'cu{}{}-torch220'
elif v  < V('2.4.0'): x = 'cu{}{}-torch230'
elif v  < V('2.5.0'): x = 'cu{}{}-torch240'
elif v  < V('2.6.0'): x = 'cu{}{}-torch250'
else: raise RuntimeError(f"Torch = {v} too new!")
x = x.format(cuda.replace(".", ""), "-ampere" if is_ampere else "")
print(f'pip install --upgrade pip && pip install "unsloth[{x}] @ git+https://github.com/unslothai/unsloth.git"')

RuntimeError: CUDA = 12.6 not supported!

In [2]:

# Imports
import datetime
from datasets import load_dataset, Dataset, DatasetDict
import unsloth
import numpy as np
import torch
from trl import DPOTrainer, DPOConfig
from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:

# Load Dataset from Hub (use the dataset names from your data pipeline)
dataset_name = "ciacco/m1_dpo_large_combined_20250609_0508"  # Updated with actual timestamp
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)

print(f"Train dataset: {len(dataset['train'])} samples")
print(f"Eval dataset: {len(dataset['eval'])} samples") 
print(f"Test dataset: {len(dataset['test'])} samples")
# Setup Unsloth Model and LoRA
print("Loading model with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-0.6B-Base",  
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    #load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False, #"unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print(f"Model loaded successfully")
print(f"CUDA available: {torch.cuda.is_available()}")

# Training Configuration
experiment_name = "unsloth_dpo_large_combined"
run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
run_name = f"{experiment_name}_{run_timestamp}"
output_dir = f"outputs/dpo_{run_name}"

print(f"Run name: {run_name}")
print(f"Output directory: {output_dir}")


Loading dataset: ciacco/m1_dpo_large_combined_20250609_0508
Train dataset: 30068 samples
Eval dataset: 3758 samples
Test dataset: 3758 samples
Loading model with Unsloth...
==((====))==  Unsloth 2025.6.1: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA A100-SXM4-40GB MIG 3g.20gb. Num GPUs = 1. Max memory: 19.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/home/my_venvs/mnlp_m2/lib/python3.12/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
Unsloth 2025.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded successfully
CUDA available: True
Run name: unsloth_dpo_large_combined_20250609_1741
Output directory: outputs/dpo_unsloth_dpo_large_combined_20250609_1741


In [4]:

# Create DPO Trainer with Unsloth optimizations
training_args = DPOConfig(
    output_dir=output_dir,
    num_train_epochs=2,  # 2 epochs as planned
    per_device_train_batch_size=2,  # Try 2 with Unsloth optimizations
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Conservative but should work
    learning_rate=3e-6,  # Lower LR for DPO
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,  # More frequent evaluation
    save_strategy="steps",
    save_steps=50,  # Frequent saves for crash recovery
    save_total_limit=10,  # Keep more checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=42,
    run_name=run_name,
    beta=0.1,  # DPO beta parameter
    report_to="tensorboard",
    logging_dir=f"logs/{run_name}",
    max_length=2048,
    max_prompt_length=1024,
    gradient_checkpointing=True,
    dataloader_drop_last=True,
    remove_unused_columns=False,
    bf16=True,  # Better stability than fp16
    dataloader_num_workers=0,  # Avoid multiprocessing issues
    push_to_hub=True,  # Auto-push checkpoints
    hub_model_id=f"ciacco/dpo_checkpoints_{run_name}",
    hub_strategy="checkpoint",
    hub_private_repo=True,
)

# Initialize DPO trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth will handle reference model
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=tokenizer,
    processing_class=tokenizer,
)

# Enable Unsloth training mode
FastLanguageModel.for_training(model)


/home/my_venvs/mnlp_m2/lib/python3.12/site-packages/datasets/utils/_dill.py:385: DeprecationWarning: co_lnotab is deprecated, use co_lines instead.
  obj.co_lnotab,  # for < python 3.10 [not counted in args]
/home/my_venvs/mnlp_m2/lib/python3.12/site-packages/google/protobuf/internal/well_known_types.py:91: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  _EPOCH_DATETIME_NAIVE = datetime.datetime.utcfromtimestamp(0)


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024, padding_idx=151654)
        (layers): ModuleList(
          (0-1): 2 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

In [ ]:

# Train Model
print(f"Starting Unsloth DPO training: {run_name}")
print("=" * 60)
print(f"Training samples: {len(dataset['train'])}")
print(f"Evaluation samples: {len(dataset['eval'])}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print("=" * 60)

try:
    dpo_trainer.train()
    print("Training completed successfully!")
    
except Exception as e:
    print(f"Training interrupted: {e}")
    print("Checking for checkpoints...")
    
    # Try to resume from latest checkpoint
    latest_checkpoint = dpo_trainer.get_last_checkpoint(output_dir)
    if latest_checkpoint:
        print(f"Resuming from: {latest_checkpoint}")
        dpo_trainer.train(resume_from_checkpoint=latest_checkpoint)


Starting Unsloth DPO training: unsloth_dpo_large_combined_20250609_1741
Training samples: 30068
Evaluation samples: 3758
Effective batch size: 16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,068 | Num Epochs = 2 | Total steps = 3,758
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 10,092,544/600,000,000 (1.68% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss,aux_loss
50,0.691900,0.693741,0.003214,0.001493,0.482437,0.001721,-413.485474,-320.115845,-0.744739,-0.662467,0,0,0,0
100,0.685600,0.684637,0.028947,0.008401,0.565726,0.020546,-413.228119,-320.046753,-0.754971,-0.668336,No Log,No Log,No Log,No Log
150,0.662900,0.664290,0.083958,0.016223,0.618148,0.067735,-412.678009,-319.968567,-0.781449,-0.682863,No Log,No Log,No Log,No Log
200,0.634800,0.639306,0.161930,0.026551,0.647951,0.135379,-411.898254,-319.865295,-0.804779,-0.692837,No Log,No Log,No Log,No Log
250,0.616800,0.617253,0.233462,0.028904,0.670037,0.204558,-411.182953,-319.841736,-0.826070,-0.702554,No Log,No Log,No Log,No Log
300,0.605500,0.597410,0.283337,0.009806,0.674029,0.273530,-410.684204,-320.032684,-0.840275,-0.707741,No Log,No Log,No Log,No Log
350,0.611600,0.579649,0.344325,0.004025,0.693454,0.340300,-410.074371,-320.090546,-0.831071,-0.694419,No Log,No Log,No Log,No Log
400,0.576100,0.567288,0.399058,0.002979,0.693720,0.396080,-409.526978,-320.101013,-0.830184,-0.689354,No Log,No Log,No Log,No Log
450,0.563100,0.556626,0.436847,-0.009831,0.701171,0.446678,-409.149139,-320.229065,-0.837922,-0.694027,No Log,No Log,No Log,No Log
500,0.544000,0.544199,0.472517,-0.034629,0.703566,0.507146,-408.792419,-320.477112,-0.835320,-0.688411,No Log,No Log,No Log,No Log


In [6]:

# Save and Push Final Model
print("Saving final model...")

# Save LoRA model locally
dpo_trainer.save_model()

# Merge LoRA with base model
print("Merging LoRA adapters...")
merged_model = model.merge_and_unload()

# Save merged model
final_model_name = f"ciacco/MNLP_M3_unsloth_dpo_{run_timestamp}"
lora_model_name = f"ciacco/MNLP_M3_unsloth_dpo_lora_{run_timestamp}"

try:
    # Push merged model
    merged_model.push_to_hub(final_model_name, private=True)
    tokenizer.push_to_hub(final_model_name, private=True)
    print(f"✅ Merged model pushed: {final_model_name}")
    
    # Push LoRA adapters separately
    model.push_to_hub(lora_model_name, private=True)
    print(f"✅ LoRA adapters pushed: {lora_model_name}")
    
except Exception as e:
    print(f"Error pushing to hub: {e}")
    print(f"Models saved locally in: {output_dir}")

print(f"\n🎉 Training completed!")
print(f"📊 Merged model: {final_model_name}")
print(f"🔧 LoRA adapters: {lora_model_name}")
print(f"📈 Training logs: {output_dir}")
print(f"📊 TensorBoard: tensorboard --logdir logs/")

Saving final model...


Uploading...:   0%|          | 0.00/51.9M [00:00<?, ?B/s]

Merging LoRA adapters...


/home/my_venvs/mnlp_m2/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


README.md:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

Uploading...:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Merged model pushed: ciacco/MNLP_M3_unsloth_dpo_20250609_1741


README.md:   0%|          | 0.00/599 [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

Saved model to https://huggingface.co/ciacco/MNLP_M3_unsloth_dpo_lora_20250609_1741
✅ LoRA adapters pushed: ciacco/MNLP_M3_unsloth_dpo_lora_20250609_1741

🎉 Training completed!
📊 Merged model: ciacco/MNLP_M3_unsloth_dpo_20250609_1741
🔧 LoRA adapters: ciacco/MNLP_M3_unsloth_dpo_lora_20250609_1741
📈 Training logs: outputs/dpo_unsloth_dpo_large_combined_20250609_1741
📊 TensorBoard: tensorboard --logdir logs/


---

In [2]:

# Imports
import datetime
from datasets import load_dataset, Dataset, DatasetDict
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig


In [ ]:

# Data Pipeline Function
def create_clean_dpo_pipeline():
    """
    Create a clean DPO training pipeline with proper splits and filtering
    """
    
    # 1. Load and combine datasets
    print("Loading datasets...")
    
    # Load your M1 dataset (smaller, domain-specific)
    m1_dpo = load_dataset("ciacco/m1_dpo_filtered_highest_total", split="train")
    print(f"M1 dataset: {len(m1_dpo)} samples")
    
    # Load juicy mix (larger, general)
    juicy_mix = load_dataset("ciacco/m1_dpo_juicy_mix", split="train")
    print(f"Juicy mix dataset: {len(juicy_mix)} samples")
    
    # 2. Standardize M1 dataset format (add missing fields and fix MCQ prompts)
    def standardize_m1_sample(sample):
        # Add question options to MCQ prompts if missing
        prompt = sample['prompt']
        if sample.get('question_type') == 'mcq' and sample.get('question_options'):
            if "Options:" not in prompt and sample['question_options']:
                prompt = f"{prompt}\n\nOptions:\n{sample['question_options']}"
        
        return {
            'prompt': prompt,
            'chosen': sample['chosen'],
            'rejected': sample['rejected'],
            'source': 'm1_dataset'
        }
    
    # Apply standardization to M1 dataset
    m1_standardized = m1_dpo.map(standardize_m1_sample)
    m1_standardized = m1_standardized.select_columns(['prompt', 'chosen', 'rejected', 'source'])
    
    # 4. Downsample juicy mix to manageable size
    juicy_sample_size = min(10000, len(juicy_mix))  # Sample 10k from juicy mix
    juicy_sampled = juicy_mix.shuffle(seed=42).select(range(juicy_sample_size))
    
    # Ensure juicy mix has source column
    if 'source' not in juicy_sampled.column_names:
        juicy_sampled = juicy_sampled.add_column('source', ['juicy_mix'] * len(juicy_sampled))
    
    print(f"Sampled juicy mix: {len(juicy_sampled)} samples")
    
    # 5. Combine datasets
    # Option 1: M1 only (for domain-specific training)
    m1_only = m1_standardized
    
    # Option 2: Combined dataset with balanced sampling
    # Take all M1 + sampled juicy mix
    combined_list = list(m1_standardized) + list(juicy_sampled)
    combined_dataset = Dataset.from_list(combined_list)
    
    print(f"Combined dataset: {len(combined_dataset)} samples")
    
    # 6. Create train/eval splits for both datasets
    def create_splits(dataset, test_size=0.1, seed=42):
        """Create train/eval splits"""
        dataset = dataset.shuffle(seed=seed)
        split_idx = int(len(dataset) * (1 - test_size))
        
        train_dataset = dataset.select(range(split_idx))
        eval_dataset = dataset.select(range(split_idx, len(dataset)))
        
        return DatasetDict({
            'train': train_dataset,
            'eval': eval_dataset
        })
    
    # Create splits
    m1_splits = create_splits(m1_only)
    combined_splits = create_splits(combined_dataset)
    
    print(f"M1 splits - Train: {len(m1_splits['train'])}, Eval: {len(m1_splits['eval'])}")
    print(f"Combined splits - Train: {len(combined_splits['train'])}, Eval: {len(combined_splits['eval'])}")
    
    # 7. Push to hub
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    
    # Push M1 only splits
    m1_splits.push_to_hub(f"ciacco/m1_dpo_clean_splits_{timestamp}", private=True)
    
    # Push combined splits  
    combined_splits.push_to_hub(f"ciacco/m1_dpo_combined_splits_{timestamp}", private=True)
    
    return m1_splits, combined_splits, timestamp


In [4]:

# Run Data Pipeline
m1_splits, combined_splits, timestamp = create_clean_dpo_pipeline()


Loading datasets...
M1 dataset: 7584 samples
Juicy mix dataset: 40630 samples


Map:   0%|          | 0/7584 [00:00<?, ? examples/s]

Sampled juicy mix: 10000 samples
Combined dataset: 17584 samples
M1 splits - Train: 6825, Eval: 759
Combined splits - Train: 15825, Eval: 1759


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


In [5]:

# Load Model and Tokenizer
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B-Base",     
                                          torch_dtype="auto",
                                          device_map="auto", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B-Base",
                                            torch_dtype="auto",
                                            device_map="auto", trust_remote_code=True)

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model memory footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"CUDA available: {torch.cuda.is_available()}")


Loading model and tokenizer...
Model memory footprint: 1.19 GB
CUDA available: True


In [6]:

# Setup Training Configuration
# Choose dataset: 'm1_only' or 'combined'
dataset_choice = 'combined' #'m1_only'  # Change this to 'combined' if you want both datasets

if dataset_choice == 'm1_only':
    train_dataset = m1_splits['train']
    eval_dataset = m1_splits['eval']
    experiment_name = "m1_clean_only"
else:
    train_dataset = combined_splits['train']
    eval_dataset = combined_splits['eval']
    experiment_name = "m1_combined_clean"

print(f"Using {dataset_choice} dataset")
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Eval dataset: {len(eval_dataset)} samples")

# Create experiment name
run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
run_name = f"{experiment_name}_{run_timestamp}"
output_dir = f"outputs/dpo_{run_name}"

print(f"Run name: {run_name}")
print(f"Output directory: {output_dir}")


Using combined dataset
Train dataset: 15825 samples
Eval dataset: 1759 samples
Run name: m1_combined_clean_20250608_1531
Output directory: outputs/dpo_m1_combined_clean_20250608_1531


In [10]:

# Create DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-7,
        logging_steps=25,
        eval_steps=100,
        eval_strategy="steps",
        save_strategy="steps",
        save_steps=200,
        save_total_limit=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=output_dir,
        run_name=run_name,
        beta=0.1,
        report_to="tensorboard",
        max_length=2048,
        max_prompt_length=1024,
        gradient_checkpointing=True,
        dataloader_drop_last=True,
        remove_unused_columns=False,
    ),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)


Extracting prompt in train dataset:   0%|          | 0/15825 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/15825 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15825 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1759 [00:00<?, ? examples/s]

In [ ]:

# Train Model
print(f"Starting training: {run_name}")
print("=" * 50)
dpo_trainer.train()


Starting training: m1_combined_clean_20250608_1531


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
100,0.679200,0.696834,0.002768,0.003606,0.286526,-0.000837,-377.831665,-322.903564,-1.305897,-1.093270
200,0.696700,0.697869,-0.000051,0.001914,0.291643,-0.001967,-377.859802,-322.920471,-1.305713,-1.093148
300,0.699900,0.694220,0.001525,-0.003653,0.302445,0.005178,-377.844055,-322.976135,-1.305758,-1.093047
400,0.694400,0.691686,0.007642,-0.002259,0.301876,0.009904,-377.782928,-322.962189,-1.304996,-1.092781
500,0.693100,0.692015,0.010461,0.000517,0.316089,0.009950,-377.754761,-322.934448,-1.303283,-1.091069
600,0.685900,0.691983,0.012075,0.001250,0.317226,0.010829,-377.738678,-322.927124,-1.302198,-1.090286
700,0.692500,0.688904,0.014334,-0.000943,0.328596,0.015284,-377.716064,-322.949036,-1.300774,-1.088575
800,0.670500,0.687389,0.019580,0.001167,0.342240,0.018416,-377.663666,-322.927948,-1.300545,-1.088745
900,0.687000,0.685968,0.022282,0.001257,0.338829,0.021034,-377.636688,-322.927063,-1.299848,-1.087616
1000,0.693400,0.687527,0.021674,0.002642,0.358158,0.019035,-377.642761,-322.913208,-1.298497,-1.086370


In [ ]:
# Save and Upload Model
print("Training completed! Saving model...")

# Save final model
final_model_name = f"ciacco/MNLP_M3_dpo_{run_name}"
model.push_to_hub(final_model_name, private=True)
tokenizer.push_to_hub(final_model_name, private=True)

print(f"Model saved as: {final_model_name}")
print(f"Training logs available in: {output_dir}")
print("Check TensorBoard logs with: tensorboard --logdir outputs/")



## Recommended Workflow:

1. **Create the new notebook**: `Clean_DPO_Pipeline.ipynb`
2. **Run cells 1-4**: This creates and uploads the clean datasets
3. **Run cells 5-6**: Load model and choose dataset configuration
4. **Run cell 7**: Create trainer (check configuration first)
5. **Run cell 8**: Start training
6. **Run cell 9**: Save the final model

## Benefits of this approach:

- **Clean separation**: Keep your old experiments intact while having a clean new pipeline
- **Modular**: Each cell has a specific purpose
- **Reproducible**: Timestamped runs and consistent naming
- **Flexible**: Easy to switch between M1-only and combined datasets
- **Proper evaluation**: Built-in train/eval splits and monitoring

This gives you a fresh start with all the improvements while keeping your previous work as reference.